# PDF to Markdown Converter for Google Colab
This notebook converts PDF files to Markdown format with high accuracy using multiple methods.

## Step 0: Configuration


In [ ]:
# ===============================================================
# CONFIG - EDIT ONLY THIS CELL, THEN RUNTIME > RUN ALL
# ===============================================================
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

# Choose input mode:
# - "upload": choose PDF file(s) directly; outputs auto-download in Colab.
# - "drive": read PDF file(s) from DRIVE_INPUT_DIR; outputs are saved to DRIVE_OUTPUT_DIR.
INPUT_MODE = "upload"
AUTO_DOWNLOAD = True

# Used for upload mode. In Colab this should normally stay /content.
DEFAULT_OUTPUT_DIR = Path("/content" if IN_COLAB else "./converted_markdown")

# Google Drive folder mode. Change these to your real Drive paths, then use INPUT_MODE="drive".
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/input_pdfs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/output_markdown")
RECURSIVE_DRIVE_SEARCH = True
PRESERVE_DRIVE_SUBFOLDERS = True

CONVERSION_CONFIG = {
    # Options: "auto", "pymupdf4llm", "pdfplumber", "fitz", "ocr".
    "method": "auto",
    "preferred_method": "pymupdf4llm",
    "ocr_language": "vie+eng",
    "ocr_dpi": 200,
    "ocr_max_pages": None,
    "scan_detection_pages": 3,
    "scan_min_chars_per_page": 30,
    "min_output_chars": 50,
}

SUPPORTED_INPUT_SUFFIXES = {".pdf"}


## Step 1: Install Required Libraries

In [ ]:
!pip install pymupdf4llm pymupdf pillow pytesseract pdfplumber -q

import platform
import shutil
import subprocess


def install_tesseract_language_data_if_needed() -> None:
    if platform.system() != "Linux" or shutil.which("apt-get") is None:
        return

    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "tesseract-ocr-vie"], check=True)


def installed_tesseract_languages() -> set[str]:
    result = subprocess.run(
        ["tesseract", "--list-langs"],
        capture_output=True,
        text=True,
        check=True,
    )
    return {
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip() and not line.lower().startswith("list of available")
    }


if shutil.which("tesseract") is None:
    install_tesseract_language_data_if_needed()
else:
    print("Tesseract is already installed.")

if shutil.which("tesseract") is None:
    print("Tesseract is not installed. Install it locally, for example: brew install tesseract tesseract-lang")
else:
    languages = installed_tesseract_languages()
    if "vie" not in languages:
        install_tesseract_language_data_if_needed()
        languages = installed_tesseract_languages()
    if "vie" not in languages:
        print("Vietnamese Tesseract language data is missing. Install it locally, for example: brew install tesseract-lang")
    else:
        print("Vietnamese Tesseract language data is available.")


## Step 2: Import Libraries

In [ ]:
import io
import os
from pathlib import Path
from typing import Optional

import fitz
import pdfplumber
import pymupdf4llm
import pytesseract
from PIL import Image

input_files = []
output_dir = DEFAULT_OUTPUT_DIR
input_root = None
converted_files = []


## Step 3: Upload PDF Files (Optional)

In [ ]:
input_files = []
output_dir = DEFAULT_OUTPUT_DIR
input_root = None


def iter_input_files(input_dir: Path, suffixes: set[str], recursive: bool = True) -> list[Path]:
    pattern = "**/*" if recursive else "*"
    return [
        path for path in sorted(input_dir.glob(pattern))
        if path.is_file() and path.suffix.lower() in suffixes
    ]


def output_path_for(input_file: Path, output_root: Path, input_root: Path | None = None) -> Path:
    if INPUT_MODE == "drive" and PRESERVE_DRIVE_SUBFOLDERS and input_root is not None:
        try:
            relative_parent = input_file.parent.relative_to(input_root)
            target_dir = output_root / relative_parent
        except ValueError:
            target_dir = output_root
    else:
        target_dir = output_root
    target_dir.mkdir(parents=True, exist_ok=True)
    return target_dir / f"{input_file.stem}.md"


def validate_unique_output_paths(input_paths: list[Path], output_root: Path, input_root: Path | None = None) -> None:
    seen = {}
    for input_path in input_paths:
        output_path = output_path_for(input_path, output_root, input_root)
        key = str(output_path.resolve() if output_path.exists() else output_path.absolute())
        if key in seen:
            raise ValueError(
                f"Duplicate output path would be created for {seen[key]} and {input_path}: {output_path}. "
                "Set PRESERVE_DRIVE_SUBFOLDERS=True or rename one input file."
            )
        seen[key] = input_path


if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Direct file upload is available in Google Colab only. Use INPUT_MODE='drive' in Colab, or run this notebook in Colab.")

    output_dir.mkdir(parents=True, exist_ok=True)
    print("Choose PDF file(s) from your computer:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        file_path = Path("/content") / filename
        if file_path.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
            raise ValueError(f"Uploaded file is not a PDF: {filename}")
        input_files.append(file_path)

    if not input_files:
        raise ValueError("No PDF file was uploaded.")

    print()
    print(f"Selected {len(input_files)} PDF file(s):")
    for file_path in input_files:
        print(f"- {file_path.name} ({file_path.stat().st_size / 1024 / 1024:.2f} MB)")
elif INPUT_MODE == "drive":
    print("INPUT_MODE='drive': direct upload skipped; Drive folder will be mounted later.")
else:
    raise ValueError("INPUT_MODE must be either 'upload' or 'drive'.")


## Step 4: Mount Google Drive Folder Mode (Optional)

In [ ]:
if INPUT_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Google Drive folder mode is available in Google Colab only.")

    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    input_root = DRIVE_INPUT_DIR
    input_files = iter_input_files(DRIVE_INPUT_DIR, SUPPORTED_INPUT_SUFFIXES, RECURSIVE_DRIVE_SEARCH)
    output_dir = DRIVE_OUTPUT_DIR

    if not input_files:
        raise FileNotFoundError(f"No PDF files found in: {DRIVE_INPUT_DIR}")

    print("Google Drive mounted successfully.")
    print(f"Input folder: {DRIVE_INPUT_DIR}")
    print(f"Output folder: {DRIVE_OUTPUT_DIR}")
    print(f"Found {len(input_files)} PDF file(s).")
else:
    print("INPUT_MODE='upload': Google Drive folder mode skipped.")


## Step 5: Define Conversion Functions

In [ ]:
def save_markdown(md_text: str, output_path: Optional[str] = None) -> None:
    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(md_text)
        print(f"✓ Saved to {output_path}")


def normalize_markdown_table_cell(value) -> str:
    if value is None:
        return ""
    return str(value).replace("\n", "<br>").replace("|", "\\|").strip()


def is_scanned_or_image_only_pdf(pdf_path: str, config: dict = CONVERSION_CONFIG) -> bool:
    sample_pages = config.get("scan_detection_pages", 3)
    min_chars_per_page = config.get("scan_min_chars_per_page", 30)

    with fitz.open(pdf_path) as doc:
        pages_to_check = min(len(doc), sample_pages)
        if pages_to_check == 0:
            return True

        extracted_chars = 0
        for page_index in range(pages_to_check):
            extracted_chars += len(doc[page_index].get_text("text").strip())

    avg_chars = extracted_chars / pages_to_check
    print(f"Auto-detect: average extractable text = {avg_chars:.0f} chars/page")
    return avg_chars < min_chars_per_page


def convert_pdf_to_md_pymupdf4llm(pdf_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert PDF to Markdown using pymupdf4llm (best for structured documents).
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Optional path to save the markdown file
    
    Returns:
        Markdown content as string
    """
    try:
        md_text = pymupdf4llm.to_markdown(pdf_path)
        save_markdown(md_text, output_path)
        return md_text
    except Exception as e:
        print(f"✗ Error with pymupdf4llm: {str(e)}")
        return ""


def convert_pdf_to_md_pdfplumber(pdf_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert PDF to Markdown using pdfplumber (best for tables and structured data).
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Optional path to save the markdown file
    
    Returns:
        Markdown content as string
    """
    try:
        md_content = []
        
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                md_content.append(f"## Page {page_num}\n")
                
                # Extract text
                text = page.extract_text()
                if text:
                    md_content.append(text)
                    md_content.append("\n")
                
                # Extract tables
                tables = page.extract_tables()
                for table in tables or []:
                    rows = [row for row in table if row]
                    if not rows:
                        continue

                    column_count = max(len(row) for row in rows)
                    normalized_rows = [list(row) + [None] * (column_count - len(row)) for row in rows]
                    header = [normalize_markdown_table_cell(cell) for cell in normalized_rows[0]]

                    md_content.append("\n| " + " | ".join(header) + " |\n")
                    md_content.append("| " + " | ".join(["---"] * column_count) + " |\n")
                    for row in normalized_rows[1:]:
                        cells = [normalize_markdown_table_cell(cell) for cell in row]
                        md_content.append("| " + " | ".join(cells) + " |\n")
                    md_content.append("\n")
        
        md_text = "".join(md_content)
        save_markdown(md_text, output_path)
        return md_text
    except Exception as e:
        print(f"✗ Error with pdfplumber: {str(e)}")
        return ""


def convert_pdf_to_md_fitz(pdf_path: str, output_path: Optional[str] = None) -> str:
    """
    Convert PDF to Markdown using PyMuPDF (best for complex layouts).
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Optional path to save the markdown file
    
    Returns:
        Markdown content as string
    """
    try:
        md_content = []

        with fitz.open(pdf_path) as doc:
            for page_num, page in enumerate(doc, 1):
                md_content.append(f"## Page {page_num}\n")

                blocks = page.get_text("blocks")
                text_blocks = [block for block in blocks if block[6] == 0 and block[4].strip()]
                text_blocks.sort(key=lambda block: (round(block[1], 1), round(block[0], 1)))

                for block in text_blocks:
                    md_content.append(block[4].strip() + "\n\n")

        md_text = "".join(md_content)
        save_markdown(md_text, output_path)
        return md_text
    except Exception as e:
        print(f"✗ Error with PyMuPDF: {str(e)}")
        return ""


def convert_pdf_to_md_ocr(pdf_path: str, output_path: Optional[str] = None, config: dict = CONVERSION_CONFIG) -> str:
    """Convert scanned/image-only PDF pages to Markdown using Tesseract OCR."""
    try:
        md_content = []
        language = config.get("ocr_language", "vie+eng")
        dpi = config.get("ocr_dpi", 200)
        max_pages = config.get("ocr_max_pages")
        matrix = fitz.Matrix(dpi / 72, dpi / 72)

        with fitz.open(pdf_path) as doc:
            total_pages = len(doc) if max_pages is None else min(len(doc), max_pages)
            for page_index in range(total_pages):
                page = doc[page_index]
                print(f"OCR page {page_index + 1}/{total_pages}...")
                pix = page.get_pixmap(matrix=matrix, alpha=False)
                image = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(image, lang=language).strip()

                md_content.append(f"## Page {page_index + 1}\n")
                if text:
                    md_content.append(text + "\n\n")

        md_text = "".join(md_content)
        save_markdown(md_text, output_path)
        return md_text
    except Exception as e:
        print(f"✗ Error with OCR: {str(e)}")
        return ""


def convert_pdf_to_md_hybrid(pdf_path: str, output_path: Optional[str] = None, config: dict = CONVERSION_CONFIG) -> str:
    """
    Convert PDF to Markdown using the best available method.
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Optional path to save the markdown file
        config: Conversion settings. method can be "auto", "pymupdf4llm", "pdfplumber", "fitz", or "ocr".
    
    Returns:
        Markdown content as string
    """
    if not os.path.exists(pdf_path):
        print(f"✗ File not found: {pdf_path}")
        return ""
    
    method = config.get("method", "auto")
    if method == "auto":
        method = "ocr" if is_scanned_or_image_only_pdf(pdf_path, config) else config.get("preferred_method", "pymupdf4llm")

    converters = {
        "pymupdf4llm": convert_pdf_to_md_pymupdf4llm,
        "pdfplumber": convert_pdf_to_md_pdfplumber,
        "fitz": convert_pdf_to_md_fitz,
        "ocr": lambda path, output: convert_pdf_to_md_ocr(path, output, config),
    }

    if method not in converters:
        print(f"✗ Unknown method: {method}")
        return ""

    print(f"Converting {os.path.basename(pdf_path)} using {method}...")
    md_text = converters[method](pdf_path, output_path)

    if method != "ocr" and len(md_text.strip()) < config.get("min_output_chars", 50):
        print("Extracted text is too short; retrying with OCR...")
        md_text = convert_pdf_to_md_ocr(pdf_path, output_path, config)

    return md_text


## Step 6: Convert PDF

In [ ]:
if not input_files:
    raise ValueError("No input PDF files. Check INPUT_MODE and the configured input source in Step 0.")

import uuid

validate_unique_output_paths(input_files, output_dir, input_root)
converted_files = []

for pdf_file in input_files:
    output_path = output_path_for(pdf_file, output_dir, input_root)
    temp_output_path = output_path.with_name(f".{output_path.stem}.{uuid.uuid4().hex}.tmp{output_path.suffix}")

    md_content = convert_pdf_to_md_hybrid(str(pdf_file), str(temp_output_path), config=CONVERSION_CONFIG)

    if len(md_content.strip()) < CONVERSION_CONFIG.get("min_output_chars", 50):
        if temp_output_path.exists():
            temp_output_path.unlink()
        raise RuntimeError(f"Conversion produced little or no text for {pdf_file.name}. Check OCR language/config or PDF quality.")
    if not temp_output_path.exists() or temp_output_path.stat().st_size == 0:
        raise RuntimeError(f"Conversion failed or produced an empty file for {pdf_file.name}.")

    temp_output_path.replace(output_path)
    converted_files.append(output_path)
    print(f"✓ Converted: {pdf_file.name} -> {output_path} ({len(md_content)} characters)")

print()
print(f"✓ Conversion completed for {len(converted_files)} file(s).")
print(f"Output folder: {output_dir}")


## Step 7: Download Converted File

In [ ]:
if not converted_files:
    raise FileNotFoundError("No converted files found. Run the conversion cell first.")

if INPUT_MODE == "drive":
    print(f"Output files are saved in Google Drive: {output_dir}")
elif AUTO_DOWNLOAD:
    for output_path in converted_files:
        if not Path(output_path).exists() or Path(output_path).stat().st_size == 0:
            raise FileNotFoundError(f"Output file is missing or empty: {output_path}")
        files.download(str(output_path))
    print(f"✓ Downloaded {len(converted_files)} file(s).")
else:
    print(f"AUTO_DOWNLOAD=False. Output files are ready at: {output_dir}")


## Step 8: Alternative Methods (Optional)

In [ ]:
# Try another method if needed by changing CONVERSION_CONFIG in Step 0.

# For tables and structured data:
# CONVERSION_CONFIG["method"] = "pdfplumber"

# For complex layouts:
# CONVERSION_CONFIG["method"] = "fitz"

# Force OCR for scanned/image-only documents:
# CONVERSION_CONFIG["method"] = "ocr"


## Notes

- **pymupdf4llm**: Best for general documents, maintains structure and formatting
- **pdfplumber**: Best for tables and structured data extraction
- **PyMuPDF (fitz)**: Best for complex layouts and image-heavy documents
- **OCR**: Used automatically when the PDF has very little extractable text; force it with `CONVERSION_CONFIG["method"] = "ocr"`

If the default conversion is not satisfactory, change `CONVERSION_CONFIG` in Step 0 or Step 8, then rerun Step 6.